# 02 · Retrieval / RAG index  _(owner: M2)_

Build the ChromaDB glossary index over MedlinePlus + lay-health guidelines; measure retrieval hit quality; wire it into the Simplifier.

In [2]:
import pandas as pd

df = pd.read_csv("../data/train.csv")

print(df.head())
print(df.columns)

   question      pmid                                         input_text  \
0         1  29857264  Exercise-Associated Muscle Cramps (EAMC) are a...   
1         1  33722257  Background: Muscle cramp is a painful, involun...   
2         1  31696455  Muscle cramp is a temporary but intense and pa...   
3         1  30168894  Muscular cramp is a common symptom in healthy ...   
4         1  29763070  Muscle cramps result in continuous, involuntar...   

                                         target_text  Adaptation_Version  \
0  Exercise-Associated Muscle Cramps (EAMC) are a...                   2   
1  Muscle cramps are unconscious contractions of ...                   2   
2  Muscle cramp is a temporary but intense, painf...                   2   
3  Muscle cramps are common in healthy people, es...                   2   
4  Muscle cramps cause constant and unintended co...                   2   

  Question_Type  
0             C  
1             C  
2             C  
3             

In [3]:
print("ORIGINAL MEDICAL TEXT:")
print(df.loc[0, "input_text"])

print("\nSIMPLIFIED TEXT:")
print(df.loc[0, "target_text"])

ORIGINAL MEDICAL TEXT:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim of this review is to examine the recent literature, in terms of physiological mechanisms of EAMC. A comprehensive search was conducted on PubMed and Google Scholar. The following terminology was applied: muscle cramps, neuromuscular hypothesis (or thesis), dehydration hypothesis, Exercise-Associated muscle cramps, nocturnal cramps, muscle spasm, muscle fatigue. From the initial literature of 424 manuscripts, sixty-nine manuscripts were included, analyzed, compared and summa

In [4]:
for i in range(3):
    print(f"\n=== PLABA EXAMPLE {i} ===")

    print("\nORIGINAL:")
    print(df.loc[i, "input_text"][:500])

    print("\nSIMPLIFIED:")
    print(df.loc[i, "target_text"][:500])


=== PLABA EXAMPLE 0 ===

ORIGINAL:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim o

SIMPLIFIED:
Exercise-Associated Muscle Cramps (EAMC) are a common type of muscle spasm, in which a muscle continually contracts without intention, causing pain. Scientists have tried to explain why these cramps happen, but have not been able to. An idea commonly returned to throughout the years is that EAMCs may be caused by heat. Recently, though, more likely explanations are thought to include dehydration, lack of electrolytes, or issues with the nerves co

In [5]:
# Get one example from each different question group
examples = df.drop_duplicates(subset="question")

# Look at the first five different questions
for _, row in examples.head(5).iterrows():

    print(f"\n=== QUESTION {row['question']} ===")

    print("\nORIGINAL:")
    print(row["input_text"][:400])

    print("\nSIMPLIFIED:")
    print(row["target_text"][:400])


=== QUESTION 1 ===

ORIGINAL:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechan

SIMPLIFIED:
Exercise-Associated Muscle Cramps (EAMC) are a common type of muscle spasm, in which a muscle continually contracts without intention, causing pain. Scientists have tried to explain why these cramps happen, but have not been able to. An idea commonly returned to throughout the years is that EAMCs may be caused by heat. Recently, though, more likely explanations are thought to include dehydration, 

=== QUESTION 3 ===

ORIGINAL:
Hyperkalemia is a frequent clinical abnormality in patients with chronic kidney disease, and it is associated with higher r

In [6]:
medical_vocab = [
    {
        "term": "etiology",
        "aliases": ["cause of disease"],
        "definition": "The cause of a disease or medical condition."
    },
    {
        "term": "hyperkalemia",
        "aliases": ["high blood potassium"],
        "definition": "Higher-than-normal potassium in the blood."
    },
    {
        "term": "neuromuscular",
        "aliases": ["nerve and muscle"],
        "definition": "Related to nerves and muscles."
    }
]

for item in medical_vocab:
    print(item["term"], ":", item["definition"])

etiology : The cause of a disease or medical condition.
hyperkalemia : Higher-than-normal potassium in the blood.
neuromuscular : Related to nerves and muscles.


In [8]:
import chromadb
from pathlib import Path

# Find the project's data folder.
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

# Create a local database inside data/.
db_path = project_root / "data" / "chroma_db"

client = chromadb.PersistentClient(path=str(db_path))

# Create or open a vocabulary collection.
collection = client.get_or_create_collection(
    name="medical_vocabulary"
)

print("ChromaDB collection created!")


ChromaDB collection created!


In [ ]:
# Store the medical vocabulary in the collection.
collection.upsert(
    ids=[item["term"] for item in medical_vocab],
    documents=[
        item["term"] + ": " + item["definition"]
        for item in medical_vocab
    ],
    metadatas=[
        {"term": item["term"]}
        for item in medical_vocab
    ]
)

print("Stored entries:", collection.count())

/Users/andrewnguyen/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:33<00:00, 2.48MiB/s]


Stored entries: 3


In [ ]:
# Query the collection for a medical term
results = collection.query(
    query_texts=["high blood potassium"],
    n_results=1
)

print(results["documents"][0][0])

hyperkalemia: Higher-than-normal potassium in the blood.


In [ ]:
# Query the collection for multiple medical terms
queries = [
    "What causes a disease?",
    "Problems involving nerves and muscles",
    "Too much potassium in the bloodstream"
]

for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=1
    )

    print("Search:", query)
    print("Result:", results["documents"][0][0])
    print()

Search: What causes a disease?
Result: etiology: The cause of a disease or medical condition.

Search: Problems involving nerves and muscles
Result: neuromuscular: Related to nerves and muscles.

Search: Too much potassium in the bloodstream
Result: hyperkalemia: Higher-than-normal potassium in the blood.



In [12]:
additional_vocab = [
    {
        "term": "physiological mechanism",
        "definition": "How a process works inside the body."
    },
    {
        "term": "electrolyte depletion",
        "definition": "Losing too much of important minerals such as sodium or potassium."
    },
    {
        "term": "chronic kidney disease",
        "definition": "A long-term condition in which the kidneys are damaged or do not work as well as they should."
    },
    {
        "term": "malignant arrhythmias",
        "definition": "Dangerous abnormal heart rhythms that can be life-threatening."
    },
    {
        "term": "fasting plasma glucose",
        "definition": "A measurement of blood sugar after a period without eating."
    },
    {
        "term": "diagnostic criteria",
        "definition": "Medical rules or test results used to identify a condition."
    },
    {
        "term": "double-blind study",
        "definition": "A study where participants and certain researchers do not know which treatment each participant receives."
    }
]

# Combine the original and new vocabulary.
all_vocab = medical_vocab + additional_vocab

print("Total terms:", len(all_vocab))

Total terms: 10


In [18]:
collection.upsert(
    ids=[item["term"] for item in all_vocab],
    documents=[
        item["term"] + ": " + item["definition"]
        for item in all_vocab
    ]
)

print("Stored entries:", collection.count())

Stored entries: 10


In [19]:
results = collection.query(
    query_texts=["dangerous irregular heartbeat"],
    n_results = 3
)

for document in results["documents"][0]:
    print(document)

malignant arrhythmias: Dangerous abnormal heart rhythms that can be life-threatening.
etiology: The cause of a disease or medical condition.
hyperkalemia: Higher-than-normal potassium in the blood.


In [20]:
results = collection.query(
    query_texts=["dangerous irregular heartbeat"],
    n_results=3,
    include=["documents", "distances"]
)

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {document}")

Distance: 0.6661 | malignant arrhythmias: Dangerous abnormal heart rhythms that can be life-threatening.
Distance: 1.4204 | etiology: The cause of a disease or medical condition.
Distance: 1.4684 | hyperkalemia: Higher-than-normal potassium in the blood.


In [ ]:
# Embeddings:
# Convert text into numerical vectors so computers can compare meanings + identify similar words or phrases.

# ChromaDB:
# A vector database that stores text + embeddings, allowing us to search for medical definitions by meaning using it's embedding model.

# Queries:
# Search requests sent to ChromaDB to find relevant information.
# Searching "high blood potassium" can retrieve the medical term "hyperkalemia".